# 02 — Building the contract analyzer

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — Legal Tech |
| **Level** | Advanced |
| **Time** | ~40 min |
| **Prerequisites** | `01_architecture.ipynb` |
| **Open in Colab** | `labs/03_contract_analyzer/02_build.ipynb` |

> **Goal:** Implement the complete contract-analysis pipeline — from raw legal text
> to a structured risk report with negotiation recommendations.

## Setup

In [ ]:
# ── Install dependencies ──
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "chromadb>=0.4.0" "pandas>=2.0.0"

import os, json, re, warnings
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from openai import OpenAI
from sentence_transformers import SentenceTransformer

# ── API setup ──
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

if client:
    print("✓ OpenAI client ready")
else:
    print("⚠  OPENAI_API_KEY not set — LLM calls will use mock responses")

print("✓ Dependencies installed")

## 1 · Sample contracts

We create three **synthetic contracts** that represent common real-world scenarios:

1. **Fair SaaS vendor agreement** — balanced, market-standard terms
2. **Risky service agreement** — hidden one-sided clauses
3. **Employment contract** — IP assignment and non-compete concerns

Each is ~1,500 words of realistic legal text.

In [ ]:
# ── Contract 1: Fair SaaS Vendor Agreement ──
CONTRACT_FAIR_SAAS = """
SOFTWARE AS A SERVICE AGREEMENT

This Software as a Service Agreement ("Agreement") is entered into as of
January 1, 2025, by and between CloudPlatform Inc., a Delaware corporation
("Provider"), and EnterpriseCo Ltd., a New York corporation ("Customer").

1. DEFINITIONS

1.1 "Service" means the cloud-based software platform described in Exhibit A,
including all updates, patches, and maintenance releases provided during the Term.

1.2 "Customer Data" means all data uploaded, stored, or processed by Customer
through the Service, which shall remain the exclusive property of Customer.

1.3 "Confidential Information" means non-public technical, business, or financial
information disclosed by either party, excluding information that is publicly
available, independently developed, or rightfully received from third parties.

2. SERVICE AND LICENSE

2.1 Provider grants Customer a non-exclusive, non-transferable right to access and
use the Service during the Term for Customer's internal business purposes, subject
to the usage limitations in Exhibit A. Customer shall not sublicense, reverse
engineer, or create derivative works from the Service.

2.2 Provider shall maintain the Service with 99.9% uptime, measured monthly,
excluding scheduled maintenance windows communicated at least 48 hours in advance.

3. PAYMENT TERMS

3.1 Customer shall pay the annual subscription fee of $120,000, invoiced quarterly
in advance ($30,000 per quarter). Payment is due within 30 days of invoice date.

3.2 Late payments shall accrue interest at the lesser of 1.0% per month or the
maximum rate permitted by applicable law. Provider may suspend access after 60 days
of non-payment, upon 15 days prior written notice.

4. DATA PROTECTION AND SECURITY

4.1 Provider shall implement and maintain industry-standard security measures,
including encryption at rest (AES-256) and in transit (TLS 1.2+), regular
penetration testing, and SOC 2 Type II compliance.

4.2 Customer Data shall be stored in data centers located in the United States.
Provider shall not access Customer Data except as necessary to provide the Service
or as directed by Customer in writing.

5. CONFIDENTIALITY

5.1 Each party shall protect the other's Confidential Information with at least the
same degree of care it uses to protect its own confidential information, but no less
than reasonable care. This obligation continues for three (3) years after disclosure.

5.2 Standard exceptions apply: information that (a) becomes publicly available through
no fault of the receiving party, (b) was known prior to disclosure, (c) is independently
developed, or (d) must be disclosed by law with prompt notice to the disclosing party.

6. INTELLECTUAL PROPERTY

6.1 Provider retains all rights, title, and interest in the Service, including all
intellectual property rights. Nothing in this Agreement transfers ownership of the
Service or Provider's pre-existing IP to Customer.

6.2 Customer retains all rights in Customer Data. Provider is granted a limited license
to process Customer Data solely to provide the Service.

6.3 Any customizations or integrations developed by Provider specifically for Customer
under a separate Statement of Work shall be owned by Customer upon full payment.

7. INDEMNIFICATION

7.1 Provider shall indemnify, defend, and hold harmless Customer from third-party claims
alleging that the Service, as provided by Provider, infringes a valid United States patent,
copyright, or trade secret, subject to the limitation of liability in Section 8.

7.2 Customer shall indemnify Provider from claims arising from Customer's use of the
Service in violation of this Agreement or applicable law.

7.3 Indemnification obligations are conditioned on prompt notice, reasonable cooperation,
and sole control of the defense by the indemnifying party.

8. LIMITATION OF LIABILITY

8.1 NEITHER PARTY'S AGGREGATE LIABILITY UNDER THIS AGREEMENT SHALL EXCEED THE TOTAL
FEES PAID OR PAYABLE BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING THE CLAIM.

8.2 IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR INDIRECT, INCIDENTAL, CONSEQUENTIAL,
SPECIAL, OR PUNITIVE DAMAGES, INCLUDING LOST PROFITS OR LOST DATA, REGARDLESS OF THE
FORM OF ACTION OR THEORY OF LIABILITY.

8.3 The foregoing limitations shall not apply to: (a) indemnification obligations under
Section 7, (b) breach of confidentiality under Section 5, or (c) willful misconduct.

9. TERM AND TERMINATION

9.1 This Agreement shall commence on the Effective Date and continue for an initial term
of one (1) year, automatically renewing for successive one-year periods unless either
party provides written notice of non-renewal at least 60 days before the end of the
then-current term.

9.2 Either party may terminate this Agreement immediately upon material breach by the
other party that remains uncured for 30 days after written notice specifying the breach.

9.3 Upon termination, Provider shall make Customer Data available for export for 90 days,
after which Provider may delete Customer Data. Sections 5, 7, 8, and 10 survive termination.

10. DISPUTE RESOLUTION

10.1 The parties shall first attempt to resolve disputes through good-faith negotiation.
If unresolved within 30 days, disputes shall be submitted to binding arbitration under the
rules of the American Arbitration Association in New York, New York.

10.2 This Agreement shall be governed by the laws of the State of New York, without regard
to conflict of laws principles.
"""

# ── Contract 2: Risky Service Agreement (hidden clauses) ──
CONTRACT_RISKY_SERVICE = """
PROFESSIONAL SERVICES AGREEMENT

This Professional Services Agreement ("Agreement") is made as of February 15, 2025,
between MegaCorp International ("Client") and ConsultingFirm LLC ("Provider").

1. SCOPE OF SERVICES

1.1 Provider shall perform the consulting services described in each Statement of Work
("SOW") executed under this Agreement. Services shall be performed in accordance with
industry standards and applicable professional guidelines.

1.2 Provider shall assign qualified personnel to perform the Services. Client reserves the
right to approve or reject any individual assigned, and to require replacement of any
individual at any time and for any reason.

2. COMPENSATION

2.1 Client shall pay Provider the fees set forth in each SOW. Unless otherwise stated in
the SOW, payment terms are net-90 days from Client's receipt of a valid invoice.

2.2 Client may withhold payment for any deliverable that does not conform to the
specifications in the applicable SOW, in Client's sole and reasonable judgment. Disputed
amounts may be withheld pending resolution without accruing interest or penalties.

2.3 Provider shall not increase rates during the initial two-year term. Any rate increase
for renewal terms shall not exceed 2% annually.

3. INTELLECTUAL PROPERTY

3.1 All work product, deliverables, inventions, discoveries, improvements, and creative
works of any kind conceived, developed, or produced by Provider in connection with the
Services, including all intellectual property rights therein, shall be considered works
made for hire and shall be the sole and exclusive property of Client.

3.2 To the extent any work product does not qualify as a work made for hire, Provider
hereby irrevocably assigns to Client all rights, title, and interest in such work product,
including all patent, copyright, trademark, and trade secret rights worldwide, in perpetuity.

3.3 Provider further assigns to Client any pre-existing intellectual property, tools,
frameworks, or methodologies that are incorporated into or necessary for the use of any
deliverable. Provider waives all moral rights in the work product.

4. CONFIDENTIALITY

4.1 Provider shall maintain in strict confidence all information received from Client,
including business plans, customer data, financial information, technical specifications,
and any other information disclosed in connection with the Services, regardless of whether
marked as confidential. This obligation shall continue in perpetuity.

4.2 Client shall have no reciprocal confidentiality obligation with respect to any
information received from Provider.

4.3 Provider shall not disclose to Client any confidential information of third parties
and shall ensure that all work is original and does not infringe any third-party rights.

5. INDEMNIFICATION

5.1 Provider shall indemnify, defend, and hold harmless Client, its affiliates, officers,
directors, employees, and agents from and against any and all claims, damages, losses,
liabilities, costs, and expenses (including reasonable attorneys' fees) arising from or
related to: (a) Provider's performance of the Services, (b) any breach of this Agreement
by Provider, (c) any negligent or wrongful act or omission of Provider, or (d) any claim
that the work product infringes third-party intellectual property rights.

5.2 Provider's indemnification obligations under this Section are not subject to any
limitation of liability and shall survive termination of this Agreement indefinitely.

6. LIMITATION OF LIABILITY

6.1 Client's total aggregate liability under this Agreement shall not exceed the fees
actually paid to Provider in the three (3) months preceding the claim.

6.2 Provider's liability under this Agreement is unlimited, including for direct, indirect,
consequential, incidental, special, and punitive damages of every kind.

6.3 Provider waives any right to claim limitation of liability based on the nature of
damages, forseeability, or any other equitable or legal defense.

7. TERMINATION

7.1 Client may terminate this Agreement or any SOW at any time, for any reason or no
reason, upon 10 days written notice. Upon such termination, Client shall pay only for
Services satisfactorily completed prior to the termination date, as determined by Client.

7.2 Provider may not terminate this Agreement during the initial two-year term. After
the initial term, Provider may terminate with 180 days prior written notice, subject to
completion of all outstanding SOWs.

7.3 Upon termination, Provider shall immediately deliver all work in progress, whether
complete or incomplete, to Client. Provider shall have no right to retain copies.

8. NON-COMPETE AND NON-SOLICITATION

8.1 During the term and for two (2) years thereafter, Provider shall not directly or
indirectly provide services that are competitive with Client's business to any entity
in any jurisdiction where Client conducts business.

8.2 During the term and for two (2) years thereafter, Provider shall not solicit or hire
any employee or contractor of Client, or encourage any such person to leave Client's employ.

9. REPRESENTATIONS AND WARRANTIES

9.1 Provider represents and warrants that all Services shall be performed in a professional
and workmanlike manner, free from defects, and in conformity with the highest industry
standards. Provider further warrants that all deliverables shall be free from errors and
shall perform exactly as specified, without exception.

9.2 THE FOREGOING WARRANTIES ARE IN ADDITION TO, AND NOT IN LIEU OF, ANY WARRANTIES
IMPLIED BY LAW, WHICH ARE NOT DISCLAIMED.

10. GOVERNING LAW

10.1 This Agreement shall be governed by the laws of the State of Delaware. Any dispute
shall be resolved exclusively in the state or federal courts located in Wilmington,
Delaware. Provider irrevocably consents to jurisdiction and waives any objection to venue.
"""

# ── Contract 3: Employment Contract ──
CONTRACT_EMPLOYMENT = """
EMPLOYMENT AGREEMENT

This Employment Agreement ("Agreement") is entered into as of March 1, 2025, by and
between InnovateTech Corp., a California corporation ("Company"), and Jane Developer
("Employee").

1. POSITION AND DUTIES

1.1 Company hereby employs Employee as Senior Software Engineer, reporting to the
Vice President of Engineering. Employee shall perform duties as reasonably assigned
by the Company, including software development, code review, and technical mentorship.

1.2 Employee shall devote full business time and professional effort to the performance
of duties. Employee may engage in reasonable civic, charitable, and personal activities
provided they do not interfere with Employee's duties or create a conflict of interest.

2. COMPENSATION AND BENEFITS

2.1 Base Salary: Company shall pay Employee an annual base salary of $185,000, payable
in accordance with Company's standard payroll schedule, subject to applicable withholdings.

2.2 Annual Bonus: Employee shall be eligible for an annual performance bonus of up to 20%
of base salary, based on individual and company performance, at Company's discretion.

2.3 Equity: Employee shall receive a stock option grant for 50,000 shares of Company common
stock, vesting over four years with a one-year cliff, subject to the terms of the Company's
Equity Incentive Plan.

2.4 Benefits: Employee shall be eligible for Company's standard benefits package, including
health insurance, dental, vision, 401(k) with company match, and 20 days paid time off.

3. INTELLECTUAL PROPERTY

3.1 All inventions, developments, improvements, and works of authorship conceived or created
by Employee during employment, whether during or outside business hours, that relate to
Company's current or anticipated business, research, or development, shall be the sole
property of Company ("Company Inventions").

3.2 Employee agrees to promptly disclose all Company Inventions and to execute all documents
necessary to perfect Company's ownership rights, including patent applications and assignments.

3.3 This Agreement does not apply to inventions that Employee develops entirely on personal
time, without use of Company resources, and that do not relate to Company's business — as
protected under California Labor Code Section 2870.

3.4 Employee represents that Exhibit B lists all prior inventions that Employee wishes to
exclude from this Agreement.

4. CONFIDENTIALITY

4.1 Employee shall not, during or after employment, use or disclose any Confidential Information
of the Company, except as necessary in the performance of duties. "Confidential Information"
includes trade secrets, customer lists, financial data, product roadmaps, source code,
algorithms, and any information designated as confidential.

4.2 Upon termination, Employee shall return all Company property and Confidential Information,
including devices, documents, and electronic files. Employee shall permanently delete any
Company information from personal devices.

5. NON-SOLICITATION

5.1 For twelve (12) months after termination, Employee shall not directly or indirectly solicit
any Company employee or contractor to leave the Company, or solicit any customer or prospective
customer for a competing product or service.

5.2 Note: Consistent with California law, this Agreement does not include a non-compete covenant.

6. TERMINATION

6.1 Employment is at-will. Either party may terminate this Agreement at any time, for any reason,
with or without cause, upon two weeks written notice (or, at Company's election, immediate
termination with two weeks pay in lieu of notice).

6.2 Severance: If Company terminates Employee without Cause (as defined below), Employee shall
receive: (a) six months of base salary continuation, (b) COBRA premium reimbursement for six
months, and (c) acceleration of three additional months of equity vesting, contingent on
execution of a standard release agreement.

6.3 "Cause" means: (a) material breach of this Agreement, (b) conviction of a felony, (c) willful
misconduct or gross negligence, or (d) continued failure to perform duties after written notice
and a 30-day cure period.

7. DISPUTE RESOLUTION

7.1 The parties shall first attempt to resolve any dispute through informal discussion. If
unresolved within 30 days, disputes shall be submitted to binding arbitration under JAMS rules
in San Francisco, California.

7.2 Notwithstanding the above, either party may seek injunctive relief in any court of competent
jurisdiction to prevent irreparable harm, including breach of confidentiality or IP provisions.

8. GENERAL PROVISIONS

8.1 This Agreement constitutes the entire understanding between the parties regarding Employee's
employment. Any modifications must be in writing and signed by both parties.

8.2 This Agreement shall be governed by the laws of the State of California.

8.3 If any provision is found unenforceable, the remaining provisions shall continue in full
force and effect.
"""

contracts = {
    "Fair SaaS Agreement": CONTRACT_FAIR_SAAS,
    "Risky Service Agreement": CONTRACT_RISKY_SERVICE,
    "Employment Contract": CONTRACT_EMPLOYMENT,
}

for name, text in contracts.items():
    word_count = len(text.split())
    section_count = len(re.findall(r'\n\d+\.\s+[A-Z]', text))
    print(f"  📄 {name:<30} — {word_count:,} words, {section_count} sections")

print(f"\n✓ {len(contracts)} synthetic contracts ready for analysis")

## 2 · Clause segmenter

Full implementation of the clause segmenter from notebook 01, enhanced with
better header detection and multi-paragraph clause merging.

In [ ]:
@dataclass
class Clause:
    """A single clause extracted from a contract."""
    text: str
    index: int
    section_header: Optional[str] = None

def segment_contract(text: str) -> List[Clause]:
    """Segment a contract into individual clauses.

    Uses structural cues: section numbers, paragraph breaks, legal headings.
    Merges short fragments into preceding clauses.
    """
    text = re.sub(r'\r\n', '\n', text.strip())

    # Split on section patterns
    section_re = re.compile(
        r'\n\s*(?='
        r'\d+\.\d*\s+[A-Z]'        # "1.1 Definitions" or "3. PAYMENT"
        r'|Section\s+\d+'             # "Section 3"
        r'|ARTICLE\s+[IVXLCDM]+'      # "ARTICLE IV"
        r')',
        re.MULTILINE
    )

    raw_segments = section_re.split(text)
    clauses: List[Clause] = []
    idx = 0

    for segment in raw_segments:
        # Further split on double newlines
        paragraphs = re.split(r'\n\s*\n', segment.strip())
        for para in paragraphs:
            para = para.strip()
            if not para or len(para) < 30:
                if clauses and para:
                    clauses[-1] = Clause(
                        text=clauses[-1].text + " " + para,
                        index=clauses[-1].index,
                        section_header=clauses[-1].section_header,
                    )
                continue

            # Extract header
            hdr_match = re.match(
                r'^(\d+\.\d*\s+[A-Z][A-Z ]+|[A-Z][A-Z ]{4,})',
                para
            )
            header = hdr_match.group(0).strip() if hdr_match else None

            clauses.append(Clause(text=para, index=idx, section_header=header))
            idx += 1

    return clauses

# ── Test on all 3 contracts ──
for name, text in contracts.items():
    clauses = segment_contract(text)
    print(f"\n{'='*70}")
    print(f"📄 {name} — {len(clauses)} clauses")
    print(f"{'='*70}")
    for c in clauses[:5]:  # Show first 5
        hdr = f" [{c.section_header}]" if c.section_header else ""
        short = c.text[:90].replace('\n', ' ') + "…" if len(c.text) > 90 else c.text.replace('\n', ' ')
        print(f"  [{c.index:2d}]{hdr}")
        print(f"      {short}")
    if len(clauses) > 5:
        print(f"  ... and {len(clauses) - 5} more clauses")

## 3 · Clause classifier and risk scorer

We use **OpenAI structured output** to classify each clause and score its risk.
The LLM receives:
- The clause text
- Our clause taxonomy
- Risk scoring criteria

It returns structured JSON: `{type, risk_score, explanation, suggested_mitigation}`.

In [ ]:
CLASSIFICATION_SYSTEM_PROMPT = """You are an expert contract analyst. For each clause provided,
return a JSON object with these fields:
- clause_type: one of [indemnification, limitation_of_liability, ip_assignment, termination,
  confidentiality, force_majeure, non_compete, payment_terms, warranty, dispute_resolution, general]
- risk_score: integer 1-5 (1=safe/standard, 5=critical risk)
- explanation: 1-2 sentences explaining the risk assessment
- suggested_mitigation: 1 sentence suggesting how to reduce risk (or "None needed" if low risk)

Risk factors to consider:
- Asymmetric obligations (one party bears disproportionate burden)
- Unbounded liability (no caps on damages/indemnification)
- Vague or undefined terms
- One-sided termination rights
- Overreaching IP assignment (captures pre-existing IP)
- Perpetual or unreasonable obligations
"""

def classify_and_score_clause(clause_text: str) -> Dict[str, Any]:
    """Classify a clause and score its risk using OpenAI structured output."""
    if not client:
        return _mock_classify(clause_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": CLASSIFICATION_SYSTEM_PROMPT},
            {"role": "user", "content": f"Analyze this contract clause:\n\n{clause_text}"},
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    return json.loads(response.choices[0].message.content)

def _mock_classify(clause_text: str) -> Dict[str, Any]:
    """Fallback classification using keyword heuristics (no API needed)."""
    text_lower = clause_text.lower()

    # Simple keyword-based classification
    type_keywords = {
        "indemnification": ["indemnif", "hold harmless", "defend and hold"],
        "limitation_of_liability": ["liability", "liable", "aggregate", "damages"],
        "ip_assignment": ["intellectual property", "work product", "inventions", "patent", "copyright"],
        "termination": ["terminat", "cancel", "expire"],
        "confidentiality": ["confidential", "non-disclosure", "secret", "proprietary information"],
        "force_majeure": ["force majeure", "act of god", "beyond.*control"],
        "non_compete": ["non-compete", "non-solicitation", "competitive", "solicit"],
        "payment_terms": ["payment", "invoice", "fee", "compensat", "salary"],
        "warranty": ["warrant", "guarantee", "represents and warrants"],
        "dispute_resolution": ["arbitrat", "mediat", "jurisdiction", "governing law"],
    }

    clause_type = "general"
    for ctype, keywords in type_keywords.items():
        if any(kw in text_lower for kw in keywords):
            clause_type = ctype
            break

    # Heuristic risk scoring
    risk_signals = [
        ("unlimited" in text_lower or "without limitation" in text_lower, 1.5),
        ("any and all" in text_lower, 1.0),
        ("perpetuity" in text_lower or "indefinitely" in text_lower, 1.0),
        ("sole and exclusive" in text_lower, 0.8),
        ("shall not exceed" in text_lower and "$100" in text_lower, 1.2),
        ("waives" in text_lower, 0.8),
        ("irrevocably" in text_lower, 0.5),
        ("pre-existing" in text_lower and "assign" in text_lower, 1.0),
    ]
    base_score = 2.0
    for signal, bump in risk_signals:
        if signal:
            base_score += bump
    risk_score = max(1, min(5, int(round(base_score))))

    return {
        "clause_type": clause_type,
        "risk_score": risk_score,
        "explanation": f"Classified as {clause_type} with risk score {risk_score}/5 based on language analysis.",
        "suggested_mitigation": "Review with legal counsel." if risk_score >= 3 else "None needed.",
    }

# ── Classify all clauses from the risky contract ──
risky_clauses = segment_contract(CONTRACT_RISKY_SERVICE)
print(f"Analyzing {len(risky_clauses)} clauses from Risky Service Agreement...\n")

results: List[Dict[str, Any]] = []
for clause in risky_clauses:
    result = classify_and_score_clause(clause.text)
    result["clause_index"] = clause.index
    result["clause_text_preview"] = clause.text[:80].replace('\n', ' ')
    results.append(result)

# Display as table
df = pd.DataFrame(results)
display_cols = ["clause_index", "clause_type", "risk_score", "clause_text_preview"]
available_cols = [c for c in display_cols if c in df.columns]
print(df[available_cols].to_string(index=False))

high_risk = [r for r in results if r.get("risk_score", 0) >= 4]
print(f"\n🔴 High-risk clauses (score ≥ 4): {len(high_risk)}")
for r in high_risk:
    print(f"   [{r['clause_index']:2d}] {r['clause_type']:<26} Score: {r['risk_score']} — {r.get('explanation', '')[:80]}")

## 4 · Standard clause comparison with RAG

We build a **standard clause library** with 20 market-standard examples across
all clause types.  For each risky clause, we retrieve the closest standard and
generate a comparison.

In [ ]:
import chromadb
from chromadb.config import Settings as ChromaSettings

# ── Standard clause library — 20 exemplars ──
STANDARD_LIBRARY: List[Dict[str, str]] = [
    {"type": "indemnification", "text": "Each party shall indemnify the other for direct damages from material breach, capped at 12 months of fees paid."},
    {"type": "indemnification", "text": "Vendor indemnifies Client against third-party IP claims, subject to prompt notice and sole defense control."},
    {"type": "indemnification", "text": "Mutual indemnification for negligence and willful misconduct, with standard insurance requirements."},
    {"type": "limitation_of_liability", "text": "Neither party's aggregate liability exceeds total fees paid in the 12 months preceding the claim."},
    {"type": "limitation_of_liability", "text": "Exclusion of indirect, consequential, and punitive damages for both parties equally."},
    {"type": "limitation_of_liability", "text": "Liability cap does not apply to confidentiality breaches, indemnification, or willful misconduct."},
    {"type": "ip_assignment", "text": "Vendor retains pre-existing IP. Work product created specifically for Client is assigned upon full payment."},
    {"type": "ip_assignment", "text": "Client owns deliverables; Vendor retains license to methodologies and pre-existing tools."},
    {"type": "termination", "text": "Either party may terminate for convenience with 60 days notice or immediately for uncured material breach."},
    {"type": "termination", "text": "Upon termination, each party returns confidential information. Survival clauses for IP, confidentiality, liability."},
    {"type": "confidentiality", "text": "Mutual confidentiality obligations for 3-5 years after disclosure, with standard exceptions."},
    {"type": "confidentiality", "text": "Standard carve-outs: publicly available, independently developed, legally compelled disclosure with notice."},
    {"type": "payment_terms", "text": "Payment net-30 from invoice date. Late payments accrue interest at 1% per month or maximum legal rate."},
    {"type": "payment_terms", "text": "Client may dispute invoices in good faith within 15 days; undisputed amounts remain due."},
    {"type": "warranty", "text": "Services performed in professional manner consistent with industry standards. 30-day re-performance remedy."},
    {"type": "warranty", "text": "Software warranted to perform materially as documented for 12 months. Sole remedy: repair or replace."},
    {"type": "non_compete", "text": "Non-solicitation of employees for 12 months post-termination. No broad non-compete restrictions."},
    {"type": "non_compete", "text": "Reasonable non-compete limited to specific product category and defined geography for 12 months."},
    {"type": "dispute_resolution", "text": "Good-faith negotiation, then mediation, then binding arbitration under AAA rules in neutral venue."},
    {"type": "force_majeure", "text": "Excused performance for events beyond reasonable control (natural disaster, war, pandemic) with prompt notice."},
]

# ── Build ChromaDB collection ──
chroma_client = chromadb.Client(ChromaSettings(anonymized_telemetry=False))

# Delete if exists, then create fresh
try:
    chroma_client.delete_collection("standard_clauses")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="standard_clauses",
    metadata={"description": "Market-standard contract clauses"},
)

# Add documents
for i, clause in enumerate(STANDARD_LIBRARY):
    embedding = embed_model.encode([clause["text"]], show_progress_bar=False)[0].tolist()
    collection.add(
        ids=[f"std_{i}"],
        documents=[clause["text"]],
        metadatas=[{"type": clause["type"]}],
        embeddings=[embedding],
    )

print(f"✓ Standard clause library: {collection.count()} clauses indexed in ChromaDB")

def compare_to_standard(clause_text: str, n_results: int = 3) -> List[Dict]:
    """Retrieve closest standard clauses and compute deviation."""
    query_emb = embed_model.encode([clause_text], show_progress_bar=False)[0].tolist()
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    comparisons = []
    for j in range(len(results["ids"][0])):
        distance = results["distances"][0][j]
        similarity = max(0.0, 1.0 - distance)  # Convert distance to similarity
        comparisons.append({
            "standard_type": results["metadatas"][0][j]["type"],
            "standard_text": results["documents"][0][j],
            "similarity": similarity,
            "deviation": distance,
        })
    return comparisons

# ── Compare high-risk clauses from risky contract ──
print("\nComparing risky clauses against standard library:\n")
for clause in risky_clauses:
    result = classify_and_score_clause(clause.text)
    if result.get("risk_score", 0) >= 3:
        comparisons = compare_to_standard(clause.text)
        best = comparisons[0]
        print(f"  Clause [{clause.index}] — {result['clause_type']} (risk: {result['risk_score']})")
        print(f"    Closest standard: {best['standard_type']} (similarity: {best['similarity']:.3f})")
        print(f"    Standard: {best['standard_text'][:80]}…")
        print(f"    Deviation: {best['deviation']:.3f}")
        print()

## 5 · Negotiation suggestions

For each high-risk clause, we generate:
1. **Plain-English risk explanation** — what the clause actually means
2. **Suggested alternative language** — market-standard replacement
3. **Negotiation talking points** — specific arguments to make

In [ ]:
NEGOTIATION_SYSTEM_PROMPT = """You are an expert contract negotiator. Given a risky contract clause
and the market-standard version, provide negotiation guidance as JSON with these fields:
- plain_english_risk: 1-2 sentences explaining the risk in plain English (no legal jargon)
- suggested_alternative: the full text of a fair replacement clause
- talking_points: array of 3 specific negotiation arguments to make
"""

def generate_negotiation(clause_text: str, standard_text: str, clause_type: str) -> Dict:
    """Generate negotiation suggestions for a risky clause."""
    if not client:
        return _mock_negotiation(clause_text, standard_text, clause_type)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": NEGOTIATION_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Risky clause ({clause_type}):\n{clause_text}\n\n"
                f"Market standard:\n{standard_text}\n\n"
                "Provide negotiation guidance."
            )},
        ],
        response_format={"type": "json_object"},
        temperature=0.3,
    )
    return json.loads(response.choices[0].message.content)

def _mock_negotiation(clause_text: str, standard_text: str, clause_type: str) -> Dict:
    """Fallback negotiation suggestions without API."""
    suggestions = {
        "indemnification": {
            "plain_english_risk": "This clause makes one party responsible for unlimited financial losses with no cap — potentially bankrupting exposure.",
            "suggested_alternative": "Each party shall indemnify the other for direct damages arising from material breach, subject to the liability cap in Section 8.",
            "talking_points": [
                "Industry standard is mutual indemnification capped at 12 months of fees",
                "Unlimited indemnification is not insurable and creates existential risk",
                "Propose carving out only third-party IP claims with a separate, reasonable cap",
            ],
        },
        "limitation_of_liability": {
            "plain_english_risk": "The liability caps are wildly unequal — one party is capped at a trivial amount while the other has unlimited exposure.",
            "suggested_alternative": "Neither party's aggregate liability shall exceed the total fees paid in the 12 months preceding the claim. Neither party is liable for consequential damages.",
            "talking_points": [
                "Market standard is symmetric liability caps tied to contract value",
                "Asymmetric caps signal bad faith and may not survive judicial scrutiny",
                "Propose equal caps at 12 months of fees with standard consequential damage exclusion",
            ],
        },
        "ip_assignment": {
            "plain_english_risk": "This clause takes ownership of everything you create — including your pre-existing tools and IP that existed before this contract.",
            "suggested_alternative": "Client owns deliverables created specifically under this Agreement. Provider retains all pre-existing IP and grants Client a perpetual license to use it within the deliverables.",
            "talking_points": [
                "Pre-existing IP should always be carved out — this is universal in tech contracts",
                "Overreaching IP clauses deter quality vendors and invite disputes",
                "Propose a clear distinction between project-specific deliverables and pre-existing tools",
            ],
        },
        "confidentiality": {
            "plain_english_risk": "Only one party has to keep secrets, and that obligation lasts forever with zero exceptions — even if the info becomes public.",
            "suggested_alternative": "Each party shall protect the other's confidential information for 5 years, with standard exceptions for public domain, independent development, and legal compulsion.",
            "talking_points": [
                "Mutual confidentiality is standard — one-sided obligations signal imbalance",
                "Perpetual obligations are impractical and often unenforceable",
                "Standard 3-5 year term with customary exceptions is market norm",
            ],
        },
        "termination": {
            "plain_english_risk": "Only one party can walk away, while the other is locked in regardless of circumstances — even non-payment.",
            "suggested_alternative": "Either party may terminate for convenience with 60 days notice, or immediately for material breach uncured after 30 days written notice.",
            "talking_points": [
                "Mutual termination rights are fundamental to balanced agreements",
                "One-sided termination creates perverse incentives for the party with exit rights",
                "Propose symmetric termination with reasonable notice period and cure provisions",
            ],
        },
    }
    return suggestions.get(clause_type, {
        "plain_english_risk": f"This {clause_type} clause contains terms that deviate significantly from market standards.",
        "suggested_alternative": standard_text,
        "talking_points": [
            "Request alignment with market-standard terms",
            "Propose mutual and balanced obligations",
            "Seek reasonable caps and time limitations",
        ],
    })

# ── Generate suggestions for high-risk clauses ──
print("Negotiation Suggestions for High-Risk Clauses")
print("=" * 70)

negotiation_results: List[Dict] = []
for clause in risky_clauses:
    result = classify_and_score_clause(clause.text)
    if result.get("risk_score", 0) >= 4:
        comparisons = compare_to_standard(clause.text)
        best_standard = comparisons[0]["standard_text"]
        suggestions = generate_negotiation(clause.text, best_standard, result["clause_type"])
        suggestions["clause_index"] = clause.index
        suggestions["clause_type"] = result["clause_type"]
        suggestions["risk_score"] = result["risk_score"]
        negotiation_results.append(suggestions)

        print(f"\n{'─'*70}")
        print(f"Clause [{clause.index}] — {result['clause_type'].upper()} (Risk: {result['risk_score']}/5)")
        print(f"\n  ⚠  Risk: {suggestions.get('plain_english_risk', 'N/A')}")
        print(f"\n  ✏  Suggested alternative:")
        alt = suggestions.get('suggested_alternative', 'N/A')
        print(f"     {alt[:120]}{'…' if len(alt) > 120 else ''}")
        print(f"\n  💬 Talking points:")
        for tp in suggestions.get("talking_points", []):
            print(f"     • {tp}")

print(f"\n{'='*70}")
print(f"Generated {len(negotiation_results)} negotiation recommendations")

## 6 · End-to-end demo

Complete pipeline on the **risky service agreement**: segment → classify →
score → compare → recommend → report.

In [ ]:
def analyze_contract(contract_text: str, contract_name: str = "Contract") -> Dict:
    """Full end-to-end contract analysis pipeline."""

    # Stage 1: Segment
    clauses = segment_contract(contract_text)

    # Stage 2-3: Classify and score each clause
    clause_analyses: List[Dict] = []
    for clause in clauses:
        result = classify_and_score_clause(clause.text)
        result["clause_index"] = clause.index
        result["section_header"] = clause.section_header
        result["text_preview"] = clause.text[:120].replace('\n', ' ')
        clause_analyses.append(result)

    # Stage 4: Compare high-risk clauses to standards
    deviations: List[Dict] = []
    for analysis in clause_analyses:
        if analysis.get("risk_score", 0) >= 3:
            clause_obj = clauses[analysis["clause_index"]]
            comps = compare_to_standard(clause_obj.text)
            deviations.append({
                **analysis,
                "standard_match": comps[0]["standard_text"],
                "similarity": comps[0]["similarity"],
                "deviation": comps[0]["deviation"],
            })

    # Stage 5: Generate negotiation points for top risks
    top_risks = sorted(clause_analyses, key=lambda x: x.get("risk_score", 0), reverse=True)[:5]
    negotiation_points: List[Dict] = []
    for risk in top_risks:
        if risk.get("risk_score", 0) >= 4:
            clause_obj = clauses[risk["clause_index"]]
            comps = compare_to_standard(clause_obj.text)
            suggestions = generate_negotiation(clause_obj.text, comps[0]["standard_text"], risk["clause_type"])
            negotiation_points.append({**risk, **suggestions})

    # Stage 6: Compile report
    avg_risk = np.mean([a.get("risk_score", 1) for a in clause_analyses])
    overall_level = "Low" if avg_risk < 2 else "Medium" if avg_risk < 3 else "High" if avg_risk < 4 else "Critical"
    high_count = sum(1 for a in clause_analyses if a.get("risk_score", 0) >= 4)

    report = {
        "metadata": {
            "contract_name": contract_name,
            "total_clauses": len(clauses),
            "average_risk_score": round(avg_risk, 2),
            "overall_risk_level": overall_level,
            "high_risk_clauses": high_count,
        },
        "executive_summary": (
            f"Analysis of '{contract_name}' identified {len(clauses)} clauses with an "
            f"average risk score of {avg_risk:.1f}/5.0 ({overall_level} risk). "
            f"{high_count} clauses scored 4+ and require immediate attention. "
            f"Key concerns: {', '.join(set(a['clause_type'] for a in top_risks[:3]))}."
        ),
        "clause_table": clause_analyses,
        "top_risks": top_risks,
        "deviations": deviations,
        "negotiation_points": negotiation_points,
    }
    return report

# ── Run full pipeline on the risky contract ──
print("Running full pipeline on Risky Service Agreement...\n")
report = analyze_contract(CONTRACT_RISKY_SERVICE, "MegaCorp Service Agreement")

# ── Print report ──
print("=" * 70)
print("       CONTRACT RISK ANALYSIS REPORT")
print("=" * 70)

meta = report["metadata"]
print(f"  Contract:      {meta['contract_name']}")
print(f"  Total clauses: {meta['total_clauses']}")
print(f"  Avg risk:      {meta['average_risk_score']}/5.0")
print(f"  Risk level:    {meta['overall_risk_level']}")
print(f"  High-risk:     {meta['high_risk_clauses']} clauses")

print(f"\nEXECUTIVE SUMMARY")
print(f"{'─'*70}")
print(f"  {report['executive_summary']}")

print(f"\nCLAUSE-BY-CLAUSE TABLE")
print(f"{'─'*70}")
print(f"  {'#':>3} {'Type':<26} {'Risk':>5} {'Preview'}")
for a in report["clause_table"]:
    score = a.get("risk_score", 0)
    marker = "🔴" if score >= 4 else "🟡" if score >= 3 else "🟢"
    print(f"  {a['clause_index']:>3} {a.get('clause_type','?'):<26} {marker}{score:>4}  {a['text_preview'][:50]}…")

print(f"\nTOP 5 RISKS")
print(f"{'─'*70}")
for i, risk in enumerate(report["top_risks"], 1):
    print(f"  #{i} [{risk.get('clause_type','?')}] Score: {risk.get('risk_score',0)}/5")
    print(f"     {risk.get('explanation', risk.get('text_preview',''))[:80]}")

print(f"\nNEGOTIATION RECOMMENDATIONS")
print(f"{'─'*70}")
for pt in report["negotiation_points"]:
    print(f"  ▸ {pt.get('clause_type','?').upper()} (Risk: {pt.get('risk_score',0)}/5)")
    for tp in pt.get("talking_points", [])[:2]:
        print(f"    • {tp}")
    print()

print("=" * 70)
print("✓ End-to-end analysis complete")

## Exercises

1. **Analyze the fair contract** — Run `analyze_contract(CONTRACT_FAIR_SAAS, "Fair SaaS Agreement")`.
   Compare the risk profile against the risky contract.  Are there any clauses
   that score higher than expected?  Why?

2. **Add a new clause type** — Add `data_protection` to the taxonomy and standard
   library.  Write 3 standard data-protection clauses.  Re-run the analysis on
   the fair SaaS contract (which has a data protection section).

3. **Custom risk weights** — Modify the risk scoring to weight IP and indemnification
   clauses more heavily.  How does this change the overall risk assessment?

## Key Takeaways

| # | Takeaway |
|---|---|
| 1 | Synthetic contracts enable reproducible testing without sensitive legal documents |
| 2 | Structural segmentation + LLM classification provides a robust two-stage approach |
| 3 | OpenAI structured output ensures consistent, parseable clause analysis |
| 4 | RAG-based comparison grounds risk assessment in market norms, not just abstract rules |
| 5 | Negotiation suggestions make technical analysis actionable for non-lawyers |
| 6 | The full pipeline transforms hours of manual review into minutes of automated analysis |

## What's Next

In **03 — Evaluate** we rigorously measure each component: segmentation F1,
classification accuracy, risk scoring agreement, and end-to-end quality.

## References

1. OpenAI — *Structured Outputs Guide*, https://platform.openai.com/docs/guides/structured-outputs
2. ChromaDB — *Getting Started*, https://docs.trychroma.com/getting-started
3. Harvey AI — *Contract Intelligence Platform*, https://www.harvey.ai
4. Hendrycks, D. et al. — *CUAD: An Expert-Annotated NLP Dataset for Legal Contract Review*, NeurIPS 2021